In [3]:
!pip install qiskit
!pip install qiskit_aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 92.1 MB/s eta 0:00:00


$A = Z$

$A′ = X$

$B = W := (Z+ X)/√2$

$B′= V := (Z−X)/√2$

Ici pour mesurer W, on applique ry(-π/4) avant la mesure. Et pour V, on applique ry(+π/4) avant la mesure.
La mesure se fait dans la base Z donc pour un observable dans la base X on utilise d'abord la porte d'Hadamard afin de changer de base de mesure.

In [4]:
from qiskit import *
import numpy as np
import random
from qiskit.primitives import StatevectorSampler


In [5]:
#Bell's state
qc0 = QuantumCircuit(2,2)
qc0.h(0)
qc0.cx(0,1)
print("Bell's state:")
qc0.draw()

Bell's state:


┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘
c: 2/══════════

In [7]:
# Circuit 1: Z0V1 measurement
qc1 = QuantumCircuit(2, 2)
qc1.ry(np.pi/4, 1)
qc1.measure([0, 1], [0, 1])
print("Circuit for Z0V1 measurement:")
qc1.draw()


Circuit for Z0V1 measurement:


┌─┐   
q_0: ───────────┤M├───
     ┌─────────┐└╥┘┌─┐
q_1: ┤ Ry(π/4) ├─╫─┤M├
     └─────────┘ ║ └╥┘
c: 2/════════════╩══╩═
                 0  1

In [8]:
# Circuit 2: Z0W1 measurement
qc2 = QuantumCircuit(2, 2)
qc2.ry(-np.pi/4, 1)
qc2.measure([0, 1], [0, 1])
print("Circuit for Z0W1 measurement:")
qc2.draw()

Circuit for Z0W1 measurement:


┌─┐   
q_0: ────────────┤M├───
     ┌──────────┐└╥┘┌─┐
q_1: ┤ Ry(-π/4) ├─╫─┤M├
     └──────────┘ ║ └╥┘
c: 2/═════════════╩══╩═
                  0  1

In [9]:
# Circuit 3: X0V1 measurement
qc3 = QuantumCircuit(2, 2)
qc3.h(0)
qc3.ry(np.pi/4, 1)
qc3.measure([0, 1], [0, 1])
print("Circuit for X0V1 measurement:")
qc3.draw()

Circuit for X0V1 measurement:


┌───┐   ┌─┐   
q_0: ───┤ H ├───┤M├───
     ┌──┴───┴──┐└╥┘┌─┐
q_1: ┤ Ry(π/4) ├─╫─┤M├
     └─────────┘ ║ └╥┘
c: 2/════════════╩══╩═
                 0  1

In [10]:
# Circuit 4: X0W1 measurement
qc4 = QuantumCircuit(2, 2)
qc4.h(0)
qc4.ry(-np.pi/4, 1)
qc4.measure([0, 1], [0, 1])
print("Circuit for X0W1 measurement:")
qc4.draw()

Circuit for X0W1 measurement:


┌───┐    ┌─┐   
q_0: ───┤ H ├────┤M├───
     ┌──┴───┴───┐└╥┘┌─┐
q_1: ┤ Ry(-π/4) ├─╫─┤M├
     └──────────┘ ║ └╥┘
c: 2/═════════════╩══╩═
                  0  1

In [11]:
#Random bits

a_list = []
b_list = []
for i in range(0, 1024):
  a_list.append(random.randint(0, 1))
  b_list.append(random.randint(0,1))

In [12]:
sampler = StatevectorSampler()
res = []

for i in range(len(a_list)):
    a_bit = a_list[i]
    b_bit = b_list[i]

    if a_bit == 0:
        q = qc2 if b_bit == 0 else qc1
    else:
        q = qc4 if b_bit == 0 else qc3

    job = sampler.run([qc0.compose(q)], shots=1)
    measured_value = list(job.result()[0].data.c.get_counts().keys())[0]
    out_a = int(measured_value[1])
    out_b = int(measured_value[0])
    res.append((a_bit, b_bit, out_a, out_b))

print(len(res))

1024


In [ ]:
labels = {(0, 0): "ZW", (0, 1): "ZV", (1, 0): "XW", (1, 1): "XV"}

moy = {}


print(f"    {'P(00)':>8}{'P(11)':>8}{'P(01)':>8}{'P(10)':>8}{'<AB>':>8}")
for (a, b), name in labels.items():

    n00 = 0
    n11 = 0
    n01 = 0
    n10 = 0

    # On parcourt tous les tirs et on ne garde que ceux faits avec les réglages (a, b)
    for tir in res:
        a_bit, b_bit, out_A, out_B = tir
        if a_bit == a and b_bit == b:
            if out_A == 0 and out_B == 0:
                n00 += 1
            elif out_A == 1 and out_B == 1:
                n11 += 1
            elif out_A == 0 and out_B == 1:
                n01 += 1
            else:
                n10 += 1

    n = n00 + n11 + n01 + n10


    # P(A, B)
    p00 = n00 / n
    p11 = n11 / n
    p01 = n01 / n
    p10 = n10 / n

    # Valeur moyenne <AB>
    moy[name] = p00 + p11 - p01 - p10

    print(f"{name:4}{p00:8.3f}{p11:8.3f}{p01:8.3f}{p10:8.3f}{moy[name]:8.3f}")

       P(00)   P(11)   P(01)   P(10)    <AB>
ZW     0.446   0.415   0.062   0.078   0.721
ZV     0.402   0.440   0.081   0.077   0.683
XW     0.424   0.451   0.078   0.047   0.751
XV     0.040   0.052   0.436   0.472  -0.816


In [15]:
S = moy['ZW'] + moy['ZV'] + moy['XW'] - moy['XV']
print(f"S = {S}")

S = 2.9713006786017373
